# MUC – Training und Spielleistung

Dieses Notebook liest die von `python -m ai.train` geschriebenen CSV-Dateien. Es trainiert **nicht automatisch** und kann parallel zu einem laufenden Training geöffnet werden. Nach neuen Updates: **Run → Run All Cells**.

Der voreingestellte Lauf `muc-smoke` ist ein kurzer Funktionstest, kein fertig trainierter Mitspieler. Ändere unten `RUN_NAME` auf deinen eigenen Lauf, z. B. `muc`.


In [ ]:
from pathlib import Path
from collections import Counter
import csv
import json
import math
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai" / "train.py").is_file())
RUN_NAME = "muc-smoke"  # Nach dem Trainingsstart z. B. auf "muc" ändern.
RUN_DIR = ROOT / "runs" / RUN_NAME
SMOOTHING = 5
SAVE_FIGURES = False  # Optional PNGs unter RUN_DIR / "plots" speichern.
plt.rcParams.update({"figure.figsize": (12, 7), "axes.grid": True, "grid.alpha": 0.2})

def read_csv(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

metrics = [r for r in read_csv(RUN_DIR / "metrics.csv") if r.get("total_steps")]
episodes = [r for r in read_csv(RUN_DIR / "episodes.csv") if r.get("status")]
if not metrics:
    raise RuntimeError(f"Keine Metriken unter {RUN_DIR}. Erst einen Lauf starten oder RUN_NAME ändern.")

def number(row, key):
    try:
        return float(row.get(key, ""))
    except (TypeError, ValueError):
        return float("nan")

def series(key):
    # Ein eventuell gerade geschriebener unvollständiger CSV-Datensatz wird übersprungen.
    pairs = [(number(r, "total_steps"), number(r, key)) for r in metrics]
    return np.array([(x, y) for x, y in pairs if math.isfinite(x) and math.isfinite(y)]).reshape(-1, 2)

def rolling(values, width=SMOOTHING):
    return np.array([np.mean(values[max(0, i-width+1):i+1]) for i in range(len(values))])

def plot_metric(ax, key, label=None, smooth=False, **kwargs):
    points = series(key)
    if not len(points):
        return
    x, y = points.T
    if smooth:
        ax.plot(x, y, alpha=0.2, **kwargs)
        y = rolling(y)
    ax.plot(x, y, label=label or key, marker="." if len(y) < 20 else None, **kwargs)
    ax.set_xlabel("Spielentscheidungen im Training")

def finish(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        (RUN_DIR / "plots").mkdir(parents=True, exist_ok=True)
        fig.savefig(RUN_DIR / "plots" / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()

config_path = RUN_DIR / "config.json"
config = json.loads(config_path.read_text()) if config_path.exists() else {}
print(f"Lauf: {RUN_NAME} | Updates: {len(metrics)} | Schritte: {metrics[-1]['total_steps']}")
print(f"Strecke: {config.get('airport', '?')} | Gerät: {config.get('device', '?')} | Rundereward: {config.get('round_reward', '?')}")


## Loss-Kurven

**Policy-Loss:** PPO verbessert relative Aktionswahrscheinlichkeiten. Ein stetiges Fallen auf null ist hier kein Qualitätsziel; durch Advantage-Normalisierung liegt der Wert oft ohnehin nahe null.

**Value-Loss:** Fehler der erwarteten Returns. Ein kleiner Fehler kann auch bedeuten, dass die KI zuverlässig Niederlagen vorhersagt. Entscheidend bleiben Gewinnrate und Verhalten auf separaten Partien. Die beiden Loss-Werte werden deshalb auf eigenen Skalen dargestellt.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_metric(axes[0], "policy_loss", smooth=True, color="#2563eb")
plot_metric(axes[1], "value_loss", smooth=True, color="#ea580c")
axes[0].set_title("Policy-Loss (PPO, geglättet)")
axes[1].set_title("Value-Loss (halber mittlerer quadratischer Fehler)")
finish(fig, "loss")


## Spielleistung auf Training und Validierung

Training spielt stochastisch; die periodische Validierung wählt jeweils die wahrscheinlichste Aktion. Unterschiede zwischen beiden Kurven können dadurch entstehen. Die Validierung verwendet feste, vom Training getrennte Seeds. Wiederholte Auswahl von Checkpoints anhand dieser Seeds macht sie zu **Validierungsdaten**, nicht zu einem unabhängigen Abschlusstest.

Die Intervalle sind Wilson-Intervalle über die Partien eines einzelnen Auswertungslaufs. Eine weitere erreichte Runde ist eine Diagnosehilfe und ersetzt keinen Sieg.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_metric(axes[0, 0], "train_win_rate", "Training (Sampling)", color="#94a3b8")
plot_metric(axes[0, 0], "eval_win_rate", "Validierung (Argmax)", color="#16a34a")
validation_rows = [r for r in metrics if math.isfinite(number(r, "eval_win_rate"))]
if validation_rows:
    x = [number(r, "total_steps") for r in validation_rows]
    axes[0, 0].fill_between(x, [number(r, "eval_win_ci_low") for r in validation_rows],
                           [number(r, "eval_win_ci_high") for r in validation_rows], color="#16a34a", alpha=0.12)
axes[0, 0].set(title="Gewinnrate mit 95%-Intervall", ylim=(-0.02, 1.02))
axes[0, 0].legend()
for phase, color in [("train", "#94a3b8"), ("eval", "#2563eb")]:
    plot_metric(axes[0, 1], f"{phase}_mean_round", phase, color=color)
    plot_metric(axes[1, 0], f"{phase}_mean_return", phase, color=color)
axes[0, 1].set_title("Mittlere erreichte Runde")
axes[1, 0].set_title("Mittlerer Gesamtreward")
axes[0, 1].legend()
axes[1, 0].legend()
plot_metric(axes[1, 1], "train_truncated_rate", color="#dc2626")
axes[1, 1].set(title="Abbrüche am Sicherheitslimit (sollte 0 sein)", ylim=(-0.02, 1.02))
finish(fig, "performance")


## Stabilität und Exploration

Entropie beschreibt die Streuung über erlaubte Aktionen; sie hängt auch von der Anzahl legaler Aktionen ab. KL-Divergenz und Clip-Anteil zeigen, wie stark PPO die Strategie ändert. Explained Variance bewertet die Value-Vorhersagen auf dem zuletzt gesammelten Rollout; ein negativer Wert bedeutet hier eine schlechtere Vorhersage als ein konstanter Mittelwert. Die Gradientennorm wird vor dem Clipping protokolliert.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, key, title in zip(axes.flat,
        ["entropy", "approx_kl", "clip_fraction", "explained_variance", "grad_norm", "steps_per_second"],
        ["Entropie", "Ungefähre KL-Divergenz", "PPO Clip-Anteil", "Explained Variance", "Gradientennorm vor Clipping", "Trainingsschritte / Sekunde"]):
    plot_metric(ax, key, color="#7c3aed")
    ax.set_title(title)
finish(fig, "stability")


## Verlustgründe und Spielverläufe

Links stehen die letzten bis zu 500 Trainingspartien, rechts nur die letzte vollständige Validierungsrunde. So werden frühe Trainingsfehler und wiederholt ausgewertete Seeds nicht unbemerkt zusammengezählt. Ein Spiel kann mehrere abschließende Verlustgründe haben.


In [ ]:
train_episodes = [r for r in episodes if r.get("phase") == "train"][-500:]
latest_eval_update = max((int(r["update"]) for r in episodes if r.get("phase") == "eval"), default=None)
eval_episodes = [r for r in episodes if r.get("phase") == "eval" and int(r["update"]) == latest_eval_update]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for col, (label, records) in enumerate([("Training (letzte 500)", train_episodes), ("Letzte Validierung", eval_episodes)]):
    reasons = Counter(reason for r in records for reason in r.get("reason", "").split(",") if reason)
    if reasons:
        names, counts = zip(*reasons.most_common(8))
        axes[0, col].barh(names[::-1], counts[::-1], color="#ef4444")
    else:
        axes[0, col].text(0.5, 0.5, "Keine Verlustgründe vorhanden", ha="center", transform=axes[0, col].transAxes)
    axes[0, col].set_title(f"{label}: Verlustgründe")
    rounds = [int(r["round"]) for r in records]
    axes[1, col].hist(rounds, bins=np.arange(0.5, 9.5, 1), color="#38bdf8", rwidth=0.8)
    axes[1, col].set(title=f"{label}: erreichte Runden", xlabel="Runde", ylabel="Partien")
    print(label, dict(Counter(r["status"] for r in records)))
finish(fig, "outcomes")


## Separate Tests und Vergleich mit anderen Partnern

Speichere abschließende Auswertungen im Unterordner `evaluations` deines Laufs, zum Beispiel aus dem Projektordner:

```bash
.venv/bin/python -m ai.evaluate --checkpoint runs/muc/best.pt --games 500 --json runs/muc/evaluations/self.json
.venv/bin/python -m ai.evaluate --checkpoint runs/muc/best.pt --partner heuristic --ai-role both --games 200 --json runs/muc/evaluations/heuristic-partner.json
.venv/bin/python -m ai.evaluate --baseline heuristic --games 200 --json runs/muc/evaluations/heuristic-baseline.json
```

Bei einem fremden Partner bedeutet `--ai-role both`: die angegebene Zahl Partien **pro Rolle**. Die Standard-Seeds für diese Tests beginnen bei 2.000.000; für die Trainingsvalidierung bei 1.000.000. Für belastbare Aussagen mehrere unabhängige Trainingsläufe vergleichen.


In [ ]:
reports = []
for path in sorted((RUN_DIR / "evaluations").glob("*.json")):
    data = json.loads(path.read_text())
    reports.append((path.stem, data))
    low, high = data["win_ci_low"], data["win_ci_high"]
    print(f'{path.stem}: {data["wins"]}/{data["games"]} Siege; 95%-Intervall {low:.1%}–{high:.1%}; Runde {data["mean_round"]:.2f}')
    for role in ["pilot", "kopilot"]:
        records = [r for r in data.get("episodes", []) if r["ai_role"] == role]
        if records:
            wins = sum(r["status"] == "gewonnen" for r in records)
            print(f"  {role}: {wins}/{len(records)}")
if reports:
    fig, ax = plt.subplots(figsize=(11, 4))
    for i, (name, data) in enumerate(reports):
        value = data["win_rate"]
        ax.errorbar(i, value, yerr=[[max(0, value-data["win_ci_low"])], [max(0, data["win_ci_high"]-value)]],
                    fmt="o", capsize=5, label=name)
    ax.set_xticks(range(len(reports)), [name for name, _ in reports], rotation=15, ha="right")
    ax.set(title="Abschlusstests mit 95%-Intervallen", ylabel="Gewinnrate", ylim=(-0.02, 1.02))
    finish(fig, "held-out-tests")
else:
    print("Noch keine zusätzlichen Testberichte. Die Befehle darüber erzeugen sie.")
